In [1]:
import os
import json
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.openai import OpenAI

E0000 00:00:1774168991.795248  120131 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774168991.795274  120131 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1774168991.795277  120131 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774168991.795279  120131 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1774168991.795280  120131 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


#### Set OpenAI Key

In [2]:
key = ''
try:
    with open('openai_api_key.json', 'r') as file:
        data = json.load(file)
    key = data['api_key']
except FileNotFoundError:
    print("Error: The file 'data.json' was not found. Please check the file path.")
except json.JSONDecodeError as e:
    print(f"Error: Failed to decode JSON from the file. Details: {e}")

In [3]:
# Set key
os.environ["OPENAI_API_KEY"] = key

#### Load and chunk document

In [4]:
# Load documents
documents = SimpleDirectoryReader("data").load_data()

2026-03-22 14:02:31,564 - INFO - NumExpr defaulting to 8 threads.


In [5]:
# Document Chunking
parser = SimpleNodeParser.from_defaults(chunk_size=500, chunk_overlap=50)

#### Initialize Database

In [4]:
# Initialize Chroma client (persistent)
chroma_client = chromadb.PersistentClient(path="./storage")

In [5]:
# Create or get collection
chroma_collection = chroma_client.get_or_create_collection("insurance_rag")

#### Connect Database to LlamaIndex

In [6]:
# Connect LlamaIndex to Chroma
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

#### Set up caching layer

In [11]:
# SimpleKVStore-based cache prevents re-processing nodes that haven't changed
cache = IngestionCache() 

#### Set up Ingestion pipeline with Cache

In [12]:
pipeline = IngestionPipeline(
    transformations=[
        parser,
        # LlamaIndex will automatically use default Embeddings here
    ],
    vector_store=vector_store,
    cache=cache)

In [13]:
# Run pipeline: chunks are only generated/embedded if they aren't in cache
nodes = pipeline.run(documents=documents)

#### Create Database Index

In [14]:
# Create Index from Vector Store
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

2026-03-22 14:04:12,992 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


#### Load from database if already present

In [7]:
# Load index from vector store if already present
index = VectorStoreIndex.from_vector_store(vector_store)

#### Query the RAG

In [8]:
# Use gpt-5-nano
llm = OpenAI(model="gpt-5-nano")

In [9]:
query_engine = index.as_query_engine(llm=llm)

In [10]:
response = query_engine.query("How can I renew my policy?")

In [11]:
print(response)

- Renewal happens automatically each year on the Policy Anniversary as long as the Group Policy remains in force.
- You renew by agreeing to the premium rates that are in effect on that Policy Anniversary.
- Renewal is subject to the applicable provisions (Part II, Section C) and the policy continuing in force.


In [12]:
response

Response(response='- Renewal happens automatically each year on the Policy Anniversary as long as the Group Policy remains in force.\n- You renew by agreeing to the premium rates that are in effect on that Policy Anniversary.\n- Renewal is subject to the applicable provisions (Part II, Section C) and the policy continuing in force.', source_nodes=[NodeWithScore(node=TextNode(id_='0963a9f0-76ca-4836-a4ac-c6c2f8e4762f', embedding=None, metadata={'page_label': '25', 'file_name': 'Principal-Sample-Life-Insurance-Policy.pdf', 'file_path': '/Users/avanindra/anaconda_projects/AIML/C6_GenAI/C6M17_project-rag-llamaindex/generative_doc_search2/data/Principal-Sample-Life-Insurance-Policy.pdf', 'file_type': 'application/pdf', 'file_size': 222772, 'creation_date': '2026-02-20', 'last_modified_date': '2026-02-20'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'f

In [13]:
response = query_engine.query("Summarize the main points of the document.")

In [14]:
print(response)

- Updated effective January 1, 2014.
- Document organized into major parts: Part I Definitions, Part II Policy Administration, Part III Individual Requirements and Rights.
- Part II Section A Contract covers: Entire Contract, Policy Changes, Policyholder Eligibility, Policy Incontestability (general and individual), Information to be Furnished, Certificates, Assignments, Dependent Rights, Policy Interpretation, Electronic Transactions.
- Part II Section B Premium covers: Payment Responsibility; Due Dates; Grace Period; Premium Rates; Premium Rate Changes; Premium Amount; Contributions from Members.
- Part II Section C Policy Termination covers: Failure to Pay Premium; Termination Rights of the Policyholder; Termination Rights of The Principal; Policyholder Responsibility to Members.
- Part II Section D Policy Renewal covers Renewal.
- Part III covers Individual Requirements and Rights.
- Page 4 is blank.


In [15]:
response

Response(response='- Updated effective January 1, 2014.\n- Document organized into major parts: Part I Definitions, Part II Policy Administration, Part III Individual Requirements and Rights.\n- Part II Section A Contract covers: Entire Contract, Policy Changes, Policyholder Eligibility, Policy Incontestability (general and individual), Information to be Furnished, Certificates, Assignments, Dependent Rights, Policy Interpretation, Electronic Transactions.\n- Part II Section B Premium covers: Payment Responsibility; Due Dates; Grace Period; Premium Rates; Premium Rate Changes; Premium Amount; Contributions from Members.\n- Part II Section C Policy Termination covers: Failure to Pay Premium; Termination Rights of the Policyholder; Termination Rights of The Principal; Policyholder Responsibility to Members.\n- Part II Section D Policy Renewal covers Renewal.\n- Part III covers Individual Requirements and Rights.\n- Page 4 is blank.', source_nodes=[NodeWithScore(node=TextNode(id_='7ff5eb1